In [1]:
import numpy as np
import sounddevice as sd

FS = 48_000
BIT_RATE = 50
CARRIER = 4_000
AMPLITUDE = 0.5

BITS_PER_SYMBOL = 2
SYMBOL_RATE = BIT_RATE / BITS_PER_SYMBOL
SPS = int(FS / SYMBOL_RATE)

PAYLOAD_SEC = 1.5
PAYLOAD_SYMBOLS = int(SYMBOL_RATE * PAYLOAD_SEC)

rng = np.random.default_rng(12345)
PREAMBLE_BITS = rng.integers(0, 2, size=64, dtype=int)

PAYLOAD_PATTERN = np.array([0, 0, 0, 1, 1, 1, 1, 0], dtype=int)


def bits_to_iq(b0, b1):
    if b0 == 0 and b1 == 0:
        return 1, 1
    if b0 == 0 and b1 == 1:
        return -1, 1
    if b0 == 1 and b1 == 1:
        return -1, -1
    return 1, -1


def bits_to_complex_symbols(bits):
    symbols = []

    for k in range(0, len(bits), 2):
        I, Q = bits_to_iq(bits[k], bits[k + 1])
        symbols.append((I + 1j * Q) / np.sqrt(2))

    return np.array(symbols)


def make_frame():
    payload_bits = np.resize(PAYLOAD_PATTERN, 2 * PAYLOAD_SYMBOLS)
    bits = np.concatenate([PREAMBLE_BITS, payload_bits])

    symbols = bits_to_complex_symbols(bits)
    bb = np.repeat(symbols, SPS)

    n = np.arange(len(bb))
    t = n / FS

    audio = np.real(bb * np.exp(1j * 2 * np.pi * CARRIER * t))

    audio = audio / np.max(np.abs(audio))
    audio = AMPLITUDE * audio

    return audio.astype(np.float32)


def main():
    print("QPSK TX")
    print(f"FS = {FS}")
    print(f"Bit rate = {BIT_RATE}")
    print(f"Symbol rate = {SYMBOL_RATE}")
    print(f"SPS = {SPS}")
    print(f"Carrier = {CARRIER}")
    print(f"Preamble bits = {len(PREAMBLE_BITS)}")
    print(f"Payload symbols = {PAYLOAD_SYMBOLS}")
    print("Leave this running. Start RX in another terminal.")

    frame = make_frame()

    try:
        while True:
            sd.play(frame, FS)
            sd.wait()

    except KeyboardInterrupt:
        print("\nStopped.")

make_frame()

array([0.36602542, 0.5       , 0.5       , ..., 0.36602542, 0.5       ,
       0.5       ], shape=(132480,), dtype=float32)

In [ ]:

    import numpy as np
    import sounddevice as sd
    import matplotlib.pyplot as plt

    FS = 48_000
    BIT_RATE = 50
    CARRIER = 4_000

    BITS_PER_SYMBOL = 2
    SYMBOL_RATE = BIT_RATE / BITS_PER_SYMBOL
    SPS = int(FS / SYMBOL_RATE)

    CHUNK_SEC = 5.0
    N = int(FS * CHUNK_SEC)

    LPF_WINDOW = int(SPS / 4)
    MIN_RX_AMP = 0.005

    PAYLOAD_SEC = 1.5
    PAYLOAD_SYMBOLS = int(SYMBOL_RATE * PAYLOAD_SEC)

    DISCARD_PAYLOAD_START = 2
    DISCARD_PAYLOAD_END = 4

    rng = np.random.default_rng(12345)
    PREAMBLE_BITS = rng.integers(0, 2, size=64, dtype=int)
    PREAMBLE_SYMBOLS = len(PREAMBLE_BITS) // 2

    PAYLOAD_PATTERN = np.array([0, 0, 0, 1, 1, 1, 1, 0], dtype=int)


    def moving_average(x, window):
        kernel = np.ones(window) / window
        return np.convolve(x, kernel, mode="same")


    def bits_to_iq(b0, b1):
        if b0 == 0 and b1 == 0:
            return 1, 1
        if b0 == 0 and b1 == 1:
            return -1, 1
        if b0 == 1 and b1 == 1:
            return -1, -1
        return 1, -1


    def bits_to_complex_symbols(bits):
        symbols = []

        for k in range(0, len(bits), 2):
            I, Q = bits_to_iq(bits[k], bits[k + 1])
            symbols.append((I + 1j * Q) / np.sqrt(2))

        return np.array(symbols)


    def iq_to_bits(I, Q):
        if I >= 0 and Q >= 0:
            return [0, 0]
        if I < 0 and Q >= 0:
            return [0, 1]
        if I < 0 and Q < 0:
            return [1, 1]
        return [1, 0]


    def decode_symbols(z):
        bits = []

        for val in z:
            bits.extend(iq_to_bits(np.real(val), np.imag(val)))

        return np.array(bits, dtype=int)


    def best_payload_alignment(bits_hat):
        best_errors = None
        best_expected = None
        best_shift = None

        for shift_symbols in range(4):
            shift_bits = 2 * shift_symbols
            pattern_shifted = np.roll(PAYLOAD_PATTERN, -shift_bits)
            expected = np.resize(pattern_shifted, len(bits_hat))

            errors = np.sum(bits_hat != expected)

            if best_errors is None or errors < best_errors:
                best_errors = errors
                best_expected = expected
                best_shift = shift_symbols

        return best_shift, best_errors, best_expected


    def find_preamble(z):
        known = bits_to_complex_symbols(PREAMBLE_BITS)
        best = None

        n_known = np.arange(PREAMBLE_SYMBOLS)

        max_start = len(z) - PREAMBLE_SYMBOLS - PAYLOAD_SYMBOLS

        if max_start <= 0:
            return None

        for start in range(max_start):
            test = z[start:start + PREAMBLE_SYMBOLS]

            phase_err = np.angle(test * np.conj(known))
            phase_err = np.unwrap(phase_err)

            phase_slope, phase0 = np.polyfit(n_known, phase_err, 1)

            correction = np.exp(-1j * (phase0 + phase_slope * n_known))
            test_corr = test * correction

            bits_hat = decode_symbols(test_corr)
            errors = np.sum(bits_hat != PREAMBLE_BITS)

            if best is None or errors < best["errors"]:
                best = {
                    "start": start,
                    "phase0": phase0,
                    "phase_slope": phase_slope,
                    "errors": errors,
                }

        return best


    def receive_chunk():
        print("Recording...")

        r = sd.rec(N, samplerate=FS, channels=1, dtype="float32")
        sd.wait()
        r = r[:, 0]

        rx_amp = np.max(np.abs(r))
        print("RX max amplitude:", rx_amp)

        if rx_amp < MIN_RX_AMP:
            print("Signal too weak.")
            return None

        n = np.arange(len(r))
        t = n / FS

        I_raw = 2 * r * np.cos(2 * np.pi * CARRIER * t)
        Q_raw = -2 * r * np.sin(2 * np.pi * CARRIER * t)

        I_filt = moving_average(I_raw, LPF_WINDOW)
        Q_filt = moving_average(Q_raw, LPF_WINDOW)

        best = None

        for offset in range(0, SPS, max(1, SPS // 120)):
            idx = np.arange(offset, len(r), SPS)

            z = I_filt[idx] + 1j * Q_filt[idx]

            if len(z) < PREAMBLE_SYMBOLS + PAYLOAD_SYMBOLS:
                continue

            cand = find_preamble(z)

            if cand is None:
                continue

            if best is None or cand["errors"] < best["errors"]:
                best = cand
                best["offset"] = offset
                best["z"] = z

        if best is None:
            print("Could not find preamble.")
            return None

        print("Best timing offset:", best["offset"])
        print("Preamble start symbol:", best["start"])
        print("Estimated phase0:", best["phase0"])
        print("Estimated phase slope:", best["phase_slope"])
        print("Preamble errors:", best["errors"], "/", len(PREAMBLE_BITS))
        if best["errors"] > 2:
            print("Preamble lock not clean enough; skipping chunk.")
            return None

        n_sym = np.arange(len(best["z"])) - best["start"]
        phase_track = best["phase0"] + best["phase_slope"] * n_sym

        z_corr = best["z"] * np.exp(-1j * phase_track)

        payload_start = best["start"] + PREAMBLE_SYMBOLS
        payload_z = z_corr[payload_start:payload_start + PAYLOAD_SYMBOLS]

        payload_z = payload_z[DISCARD_PAYLOAD_START:-DISCARD_PAYLOAD_END]

        if len(payload_z) == 0:
            print("No payload symbols.")
            return None

        bits_hat = decode_symbols(payload_z)

        shift, errors, bits_expected = best_payload_alignment(bits_hat)

        print("Best payload symbol shift:", shift)

        return {
            "z_payload": payload_z,
            "bits_hat": bits_hat,
            "bits_expected": bits_expected,
            "errors": errors,
            "num_bits": len(bits_hat),
        }


    def main():
        print("QPSK RX with phase-slope correction")
        print(f"FS = {FS}")
        print(f"Bit rate = {BIT_RATE}")
        print(f"Symbol rate = {SYMBOL_RATE}")
        print(f"SPS = {SPS}")
        print(f"Carrier = {CARRIER}")
        print(f"Chunk sec = {CHUNK_SEC}")
        print(f"Preamble bits = {len(PREAMBLE_BITS)}")
        print(f"Payload symbols = {PAYLOAD_SYMBOLS}")

        total_errors = 0
        total_bits = 0

        plt.ion()
        fig, ax = plt.subplots()

        try:
            while True:
                result = receive_chunk()

                if result is None:
                    continue

                total_errors += result["errors"]
                total_bits += result["num_bits"]

                Pe_chunk = result["errors"] / result["num_bits"]
                Pe_running = total_errors / total_bits

                z = result["z_payload"]

                ax.clear()
                ax.scatter(np.real(z), np.imag(z), s=30)
                ax.axhline(0)
                ax.axvline(0)
                ax.set_aspect("equal", adjustable="box")
                ax.grid(True)
                ax.set_title(f"QPSK constellation | Pe = {Pe_running:.5f}")
                ax.set_xlabel("I")
                ax.set_ylabel("Q")

                m = max(
                    np.max(np.abs(np.real(z))),
                    np.max(np.abs(np.imag(z))),
                    1e-6,
                )

                ax.set_xlim(-1.2 * m, 1.2 * m)
                ax.set_ylim(-1.2 * m, 1.2 * m)

                plt.pause(0.05)

                print(f"Chunk Pe:   {Pe_chunk:.5f}")
                print(f"Running Pe: {Pe_running:.5f}")

                print("First 40 decoded bits:")
                print(result["bits_hat"][:40])

                print("First 40 expected bits:")
                print(result["bits_expected"][:40])

        except KeyboardInterrupt:
            print("\nStopped.")

            if total_bits > 0:
                print(f"Final Pe: {total_errors / total_bits:.6f}")
                print(f"Total errors: {total_errors}")
                print(f"Total bits: {total_bits}")


    if __name__ == "__main__":
        main()
